In [ ]:
import json
import sys
from typing import List, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from plotting_utils import load_in_combined_results, load_monitoring_data
from simulate_markov_utils import plot_markov_df
from sklearn.metrics import auc, roc_curve
from turn_by_turn_monitoring import plot_markov_analysis, plot_roc_curves, plot_suspiciousness_distributions

sys.path.append('..')
from utils.analysis_utils import analyze_all_tasks, extract_verdict
from utils.path_utils import resolve_path

# Constants
MAIN_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022']
MONITOR_AGENT_MODELS = ['gpt-4o', 'claude-3-5-sonnet-20241022', 'o3-mini', 'o1', 'claude-3-5-haiku-20241022']
DEFAULT_TRANSCRIPT_TYPES = ["transcripts", "transcripts_no_hint", "transcripts_no_secondary_task"]
DOWNSAMPLE_FACTORS = [1, 2, 3, 4, 5, 10]

In [ ]:

def prepare_one_turn_debate_data(base_dir: str) -> pd.DataFrame:
    """Load and prepare the initial datasets."""
    # Load monitoring data
    one_turn_debate_df = load_monitoring_data(base_dir=base_dir)
    one_turn_debate_df['hater_first_suspiciousness'] = one_turn_debate_df.data.apply(
        lambda x: extract_verdict(x['final_response_hater_first'], tag_word='verdict')
    )
    one_turn_debate_df['supporter_first_suspiciousness'] = one_turn_debate_df.data.apply(
        lambda x: extract_verdict(x['final_response_supporter_first'], tag_word='verdict')
    )
    one_turn_debate_df['average_suspiciousness_debate'] = (one_turn_debate_df['hater_first_suspiciousness'] + one_turn_debate_df['supporter_first_suspiciousness']) / 2
    return one_turn_debate_df

def merge_and_compute_features_for_one_turn_debate(combined_results: pd.DataFrame, 
                             one_turn_debate_df: pd.DataFrame) -> pd.DataFrame:
    """Merge the dataframes and compute additional features."""
    merged_df = pd.merge(
        one_turn_debate_df,
        combined_results,
        on=['task_name', 'transcript_type', 'main_agent', 'monitor_agent', 'file_name'],
        how='left'
    ).dropna()
    
    # Compute features

    merged_df['convo_length'] = merged_df.transcript.apply(lambda x: len(x['conversation']))
    
    return merged_df


In [ ]:
if __name__ == "__main__":
    include_failed_attempts = False
    include_failed_main_task = False
    give_hint = True
    one_turn_debate_df = prepare_one_turn_debate_data(base_dir='monitor_responses/one_turn_debate')
    combined_results = load_in_combined_results(include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task)
    one_turn_debate_df['average_suspiciousness'] = (one_turn_debate_df['hater_first_suspiciousness'] + one_turn_debate_df['supporter_first_suspiciousness']) / 2
    merged_df = merge_and_compute_features_for_one_turn_debate(combined_results, one_turn_debate_df)
    plot_roc_curves(merged_df, combined_results, give_hint=give_hint, include_failed_attempts=include_failed_attempts, include_failed_main_task=include_failed_main_task, monitors = ['gpt-4o'], metrics = ['hater_first_suspiciousness', 'supporter_first_suspiciousness', 'average_suspiciousness'])
    plot_suspiciousness_distributions(merged_df, monitors = ['gpt-4o'], metrics = ['hater_first_suspiciousness', 'supporter_first_suspiciousness', 'average_suspiciousness'])